In [1]:
# !pip install langchain_mistralai

In [2]:
from langchain_mistralai import ChatMistralAI
from langchain.vectorstores import Qdrant
import qdrant_client
from qdrant_client import models
from langchain_mistralai.embeddings import MistralAIEmbeddings
from langchain.document_loaders import UnstructuredFileLoader
from langchain.chains.summarize import load_summarize_chain
from langchain.chains.question_answering import load_qa_chain


api_key = "vyhXlcO3zjBdbeHD0mmNMQf4PlBtIlM4"
model = "mistral-large-latest"
qd_host = 'http://0.0.0.0:6333'

llm = ChatMistralAI(model=model,temperature=0.5,
            max_retries=2,api_key=api_key)    

In [10]:
# # embeddings
embed = MistralAIEmbeddings(model="mistral-embed",api_key=api_key)
input_text = "The meaning of life is 42"
vector = embed.embed_query(input_text)
print(len(vector))

/Users/richardgurtsiev/Desktop/projects/save/delete_2024/del/dl_skip/semvp/.venv/lib/python3.10/site-packages/langchain_mistralai/embeddings.py:169: UserWarning: Could not download mistral tokenizer from Huggingface for calculating batch sizes. Set a Huggingface token via the HF_TOKEN environment variable to download the real tokenizer. Falling back to a dummy tokenizer that uses `len()`.
  warnings.warn(


1024


In [56]:
client = qdrant_client.QdrantClient('0.0.0.0', port=6333)

collection_name = 'collection_cv'
vectors_config = models.VectorParams(size=len(vector),distance=models.Distance.COSINE)
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=vectors_config
)

/var/folders/hk/kkj4y_8s52z072xkcbsnqr1r0000gn/T/ipykernel_14296/3300189804.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [57]:
# client.delete_collection(collection_name=collection_name)

vectore_store = Qdrant(
    client=client,
    collection_name=collection_name,
    embeddings=embed
)

In [58]:
from langchain.text_splitter import CharacterTextSplitter

def get_chunks(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )

    chunks = text_splitter.split_text(text)
    return chunks

In [59]:
with open('index.txt') as f:
    raw_text = f.read()

texts = get_chunks(raw_text)

Created a chunk of size 2133, which is longer than the specified 1000
Created a chunk of size 1528, which is longer than the specified 1000


In [72]:
print(texts[2])

Навыки: Английский — A1 — Начальный
*************************************************
Должность: Инженер-программист (программист)
Заработная плата: от 75 000 до 110 000 ₽ на руки
Локация: Location is not specified
Адрес: Москва, Озёрная, Рябиновая улица, 40с2
Тип занятости: Полная занятость, полный день
Требуемый опыт работы: 1–3 года


In [48]:
print(raw_text)

Должность: Инженер-программист
Заработная плата: от 120 000 ₽ до вычета налогов
Локация: Москва
Адрес: Raw address not available
Тип занятости: Полная занятость, удаленная работа
Требуемый опыт работы: 1–3 года
Описание:  Мы, команда компании «ИНИТИ», состоящей в реестре российских аккредитованных IT-компаний, предлагаем специалистам по профилю «инженер-программист» присоединиться к числу своих сотрудников, которым доступны все гарантии и льготы, предоставляемые работникам аккредитованных организаций, осуществляющих деятельность в области информационных технологий, в том числе отсрочка от призыва на военную службу и льготная ипотека. Обязанности:    разработка функциональных модулей высокопроизводительной системы параллельных вычислений реального времени исследование протоколов межсистемного взаимодействия, управления и мониторинга    Требования:    высшее научно-техническое образование понимание принципов построения ООП и шаблонов проектирования опыт программирования на любом из языко

In [73]:
# client.get_collection(collection_name=collection_name)
points = client.scroll(collection_name, limit=10)  # Укажите количество точек для получения
for point in points:
    print(point)
    break

[Record(id='394137ec-9503-4fdb-8942-3621d4ba28a4', payload={'page_content': 'Навыки: Английский\xa0— A1 — Начальный\n*************************************************\nДолжность: Инженер-программист (программист)\nЗаработная плата: от 75\xa0000 до 110\xa0000 ₽ на руки\nЛокация: Location is not specified\nАдрес: Москва, Озёрная, Рябиновая улица, 40с2\nТип занятости: Полная занятость, полный день\nТребуемый опыт работы: 1–3 года', 'metadata': None}, vector=None, shard_key=None, order_value=None), Record(id='6d184f1b-7fd2-4ce2-8a06-8e5b6e807afd', payload={'page_content': 'Описание: Rewatt - технологический лидер в производстве электрозарядных станций (ЭЗС) для электромобилей в России. Мы являемся резидентом Сколково и гордимся тем, что наши устройства полностью разработаны в РФ. На территории страны Rewatt уже поставил более 1000 зарядных устройств в различные регионы.Сейчас компания находится на этапе кратного роста, поэтому активно ведем подбор новых сотрудников. В том числе инженера-пр

In [64]:
# запись данных
vectore_store.add_texts(texts)

['e859d156b6784a9c919d89de28875945',
 'd8ffeba8cdf74097888c43277bf3dc4f',
 '394137ec95034fdb89423621d4ba28a4',
 '6d184f1b7fd24ce28a068e5b6e807afd',
 'a51cb96c28cb422cba8bce20d31abed4']

In [31]:
from langchain.chains import RetrievalQA


qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=vectore_store.as_retriever()
)

In [36]:
query = "Какие должности вакансии содержиться в базе?"
response = qa.invoke(query)
print(response)

{'query': 'Какие должности вакансии содержиться в базе?', 'result': 'В базе содержатся следующие вакансии:\n\n1. Программист C/C++\n2. Инженер-программист (программист)\n3. Инженер-программист'}


# Document

In [20]:
from langchain.chains import StuffDocumentsChain, LLMChain
from langchain_core.prompts import PromptTemplate

document_prompt = PromptTemplate(
    input_variables=["page_content"],
    template="{page_content}"
)
document_variable_name = "context"

prompt = PromptTemplate.from_template(
    "Summarize this content: {context}"
)
llm_chain = LLMChain(llm=llm, prompt=prompt)

chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_prompt=document_prompt,
    document_variable_name=document_variable_name
)

In [88]:
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain.chains.combine_documents.stuff import StuffDocumentsChain

stuff_prompt_override = """
Ты - ассистент HR. Ты умеешь читать резюме, анализировать их, находить нужную информацию и рекомендовывать. Ответы ты находишь в тексте новостей. 

Ответы должны быть четко структурированы и включать следующие элементы:
1. Описание характеристик.
2. Подробная информация об кандидате.
3. Насколько он или она подходят на вокансию.
---
Текст:
-----
{context}
-----
Вопрос:
{query}
"""


In [95]:
# Промпт для обработки документов
document_prompt = PromptTemplate(
            input_variables=["page_content"], template="{page_content}"
        )


# Промпт для языковой модели
document_variable_name = "context"
prompt = PromptTemplate(
    template=stuff_prompt_override, input_variables=["context", "query"]
)

llm_chain = LLMChain(llm=llm, prompt=prompt)
chain = StuffDocumentsChain(
            llm_chain=llm_chain,
            document_prompt=document_prompt,
            document_variable_name=document_variable_name,
        )

from langchain_core.documents import Document

documents = [
    Document(page_content="Марк \nХарактеристики: Сильный, умный. Долгое время работал на стройки и был сантехником.\nРаботал поваром. ", metadata={"title": "кандидат"}),
    Document(page_content="Ричард \nХарактеристики: Умный, красивый и богатый. Выстраивал ETL-системы, создавал рабочий pipeline для обработки данных. 3 года занимался разработкой.", metadata={"title": "кандидат"}),
    Document(page_content="Диана \nХарактеристики: Красивая, сексуальная и богатая.\n Долгое время работала секретаршей, и также была дизайнером.", metadata={"title": "кандидат"}),
]

input_data = {
    'input_documents': documents,
    'query': 'Какой кандидат больше подходит на позицию системного архитектора?',  # Используем оригинальный русский запрос
}
# Now, pass the 'input_data' dictionary to the 'invoke' method
response = chain.invoke(input=input_data)

# Memory and tools

In [96]:
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain.agents import Tool
from langchain.tools import BaseTool
from langchain.tools import DuckDuckGoSearchResults

In [3]:
import requests
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List, Optional

from langchain.tools import tool
from langchain.agents import AgentExecutor

from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
memory.save_context({"input": "Hi"}, {"output": "Hello! How can I help you?"})
memory.save_context({"input": "What's my name?"}, {"output": "I'm sorry, I don't have that information."})

/var/folders/hk/kkj4y_8s52z072xkcbsnqr1r0000gn/T/ipykernel_8821/3800381512.py:10: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [7]:
from langchain.memory import ConversationTokenBufferMemory

memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=20)
memory.save_context({"input": "Hi"}, {"output": "Hello! How can I assist you today?"})
memory.save_context({"input": "What's the capital of France?"}, {"output": "The capital of France is Paris."})

print(memory.load_memory_variables({}))

/Users/richardgurtsiev/Desktop/projects/save/delete_2024/del/dl_skip/semvp/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


{'history': "Human: What's the capital of France?\nAI: The capital of France is Paris."}


In [15]:
import requests
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain.tools import tool

class TripDistanceResult(BaseModel):
    start_location: str = Field(description="First location")
    end_location: str = Field(description="Second location")
    message: str = Field(description="Message about the result of the locations")

few_shot_examples = [
    {
        "request": "What are the locations between Moscow and St. Petersburg?",
        "params": {"first_city": "Moscow", "second_city": "St. Petersburg"},
    }
]

@tool(few_shot_examples)
def calculate_trip_distance(
    # first_city: str = Field(description="The first city in 'str' format"),
    # second_city: str = Field(description="The second city in 'str' format"),
     params: dict = Field(description="Parameters containing first and second cities")
) -> TripDistanceResult:
    """Find these locations between two cities."""
    first_city = params.get("first_city")
    second_city = params.get("second_city")
    base_url = f"http://127.0.0.1:8000/?first_city={first_city}&second_city={second_city}"
    try:
        response = requests.get(base_url)
        response.raise_for_status()
        data = response.json()

        return TripDistanceResult(start_location=data['start_location'], end_location=data['end_location'], message="Locations have been successfully found.")
    except requests.exceptions.RequestException as e:
        return TripDistanceResult(start_location="", end_location="", message=f"Error finding locations: {str(e)}")  # Adjusted return
    except (KeyError, IndexError):
        return TripDistanceResult(start_location="", end_location="", message="Error in server response data.")  # Adjusted return

TypeError: tool() got an unexpected keyword argument 'few_shot_examples'

In [14]:
result = calculate_trip_distance(params={"first_city": "Москва", "second_city": "Санкт-Петербург"})
print(result)

TypeError: BaseTool.__call__() got an unexpected keyword argument 'params'

In [7]:
response = requests.get('http://127.0.0.1:8000/')
response.json()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)